# PHYS 690 - Lecture 19: Diagnosing MCMC with `emcee`

## From posterior samples to trustworthy conclusions

Lecture 18 introduced `emcee` as a production ensemble sampler and used it to infer Ramsey parameters. Today we examine whether a finite `emcee` run has explored its target well enough for its posterior summaries to be reliable. We will use the package's chain and autocorrelation tools, compare independent ensembles, and connect the workflow directly to Project 2's multi-parameter cosmology inference.

## Learning goals

By the end of class, you should be able to:

1. read the dimensions of an `emcee` chain and interpret its trace plots and acceptance fractions;
2. explain autocorrelation, integrated autocorrelation time, effective sample size, and Monte Carlo error;
3. estimate these quantities using `emcee` and assess whether the estimates stabilize;
4. explain why walkers from one ensemble are not independent chains for split $\widehat R$;
5. use independently initialized ensembles to compare convergence across runs; and
6. apply these diagnostics before reporting Project 2 posterior intervals.

## Environment and project context

Select the notebook kernel associated with the virtual environment you activated for this course repository; its display name can vary. The course requirements include `emcee`. Restart the kernel after installing or updating packages.

Project 2 uses `emcee` to infer multiple correlated cosmological parameters from Pantheon+ data. The Ramsey model below is a small, fast reference problem. The parameter count and likelihood differ from Project 2, but the sampler outputs and diagnostic questions are the same. We will inspect an actual ensemble chain rather than build another sampler.

## Part 1: Define the same Ramsey posterior

For each delay, the observed excited-state count is binomial with probability

$$
p_e(t)=\frac{1}{2}\left[1+C e^{-t/\tau}\cos(\omega t+\phi)\right].
$$

We will infer $(\tau,C,\omega,\phi)$ with the same reproducible synthetic counts used in Lecture 18. This four-parameter example makes it easier to see that diagnostics are parameter-specific.

In [ ]:
# Import the data, plotting, binomial-likelihood, and emcee tools used below.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import emcee
from scipy.stats import binom

plt.rcParams.update({"figure.figsize": (8, 4.5), "axes.grid": True,
                     "grid.alpha": 0.25, "font.size": 11})

# Store known simulation values for generating data and marking reference lines.
true_parameters = {
    "tau": 2.40,
    "contrast": 0.78,
    "omega": 2.0 * np.pi * 0.85,
    "phase": 0.25,
}

# Generate the same seeded Ramsey counts as in Lecture 18.
def ramsey_probability(t, contrast, tau, omega, phase):
    t = np.asarray(t)
    p = 0.5 * (1.0 + contrast * np.exp(-t / tau) * np.cos(omega * t + phase))
    return np.clip(p, 1e-12, 1.0 - 1e-12)

data_rng = np.random.default_rng(69017)
delays = np.linspace(0.0, 5.0, 31)
shots = np.full(delays.size, 250, dtype=int)
counts = data_rng.binomial(shots, ramsey_probability(delays, **true_parameters))
ramsey_data_df = pd.DataFrame({
    "time_us": delays,
    "shots": shots,
    "excited_counts": counts,
})

# Define uniform prior bounds for the four-dimensional parameter vector.
prior_bounds = np.array([
    [0.4, 6.0],
    [0.0, 1.0],
    [2.0 * np.pi * 0.70, 2.0 * np.pi * 1.00],
    [-np.pi, np.pi],
])
parameter_names = ["tau", "contrast", "omega", "phase"]


def log_posterior(parameters):
    # Return negative infinity for proposals outside the allowed prior region.
    parameters = np.asarray(parameters)
    if np.any(parameters < prior_bounds[:, 0]) or np.any(parameters > prior_bounds[:, 1]):
        return -np.inf

    # Evaluate the binomial log likelihood; the uniform-prior constant cancels.
    tau, contrast, omega, phase = parameters
    probability = ramsey_probability(
        delays, contrast=contrast, tau=tau, omega=omega, phase=phase
    )
    return np.sum(binom.logpmf(counts, shots, probability))

In [ ]:
# Recreate the Lecture 18 view: observed excited-state fractions and the generating curve.
time_fine = np.linspace(ramsey_data_df['time_us'].min(), ramsey_data_df['time_us'].max(), 500)
ramsey_data_df['excited_fraction'] = ramsey_data_df['excited_counts'] / ramsey_data_df['shots']
ramsey_data_df['binomial_se'] = np.sqrt(
    ramsey_data_df['excited_fraction'] * (1 - ramsey_data_df['excited_fraction'])
    / ramsey_data_df['shots']
)
fig, ax = plt.subplots()
ax.errorbar(ramsey_data_df['time_us'], ramsey_data_df['excited_fraction'],
            yerr=ramsey_data_df['binomial_se'], fmt='o', ms=4,
            capsize=2, label='synthetic Ramsey data')
ax.plot(time_fine, ramsey_probability(time_fine, **true_parameters),
        lw=2, label='generating probability curve')
ax.set(xlabel='Delay time (μs)', ylabel='Excited-state fraction',
       ylim=(0, 1), title='Ramsey data and generating model')
ax.legend()
plt.show()

## Part 2: Run the production sampler and inspect traces

An `emcee` ensemble needs a starting parameter vector for **each** walker. Here the parameter order is $(\tau,C,\omega,\phi)$. We place the center at $(2.40\;\mu\mathrm{s},\;0.78,\;2\pi\times0.85\;\mathrm{rad}/\mu\mathrm{s},\;0.25\;\mathrm{rad})$, the generating values known for this synthetic example. The code sets `initial_scales = [0.15, 0.04, 0.08, 0.15]` in the same parameter order and units. These are the **standard deviations of the random offsets used to initialize walkers**, not posterior uncertainties or prior widths:

$$
\boldsymbol{\theta}^{(0)}_w
=\boldsymbol{\theta}_{\mathrm{center}}+\boldsymbol{\delta}_w,
\qquad \delta_{w,j}\sim\mathcal N(0,s_j^2).
$$

The analyst chooses the center and scales. With many walkers, generate distinct randomized starting positions so the ensemble begins as a cloud rather than with every walker at the same point. Check that all starts have finite log posterior and lie inside the prior. In a real experiment the generating parameters are unknown: use a plausible fit or physical knowledge for the center, and consider dispersed independent ensembles when checking whether other posterior regions might have been missed.


In [ ]:
# Initialize walkers near a plausible region, then verify every start is in the prior.
nwalkers = 48
ndim = len(parameter_names)
initial_center = np.array([true_parameters[name] for name in parameter_names])
initial_scales = np.array([0.15, 0.04, 0.08, 0.15])
position_rng = np.random.default_rng(1919)
initial_positions = initial_center + position_rng.normal(
    size=(nwalkers, ndim)
) * initial_scales
if not np.all(np.isfinite([log_posterior(pos) for pos in initial_positions])):
    raise ValueError("An initial walker is outside the prior bounds.")

# Burn in, reset stored samples, and retain production draws only.
np.random.seed(1920)
emcee_sampler = emcee.EnsembleSampler(nwalkers, ndim, log_posterior)
burnin_steps = 1000
production_steps = 5000
burnin_state = emcee_sampler.run_mcmc(initial_positions, burnin_steps)
emcee_sampler.reset()
emcee_sampler.run_mcmc(burnin_state, production_steps)

# Keep the step, walker, and parameter axes for diagnostics and trace plots.
production_chain = emcee_sampler.get_chain()
print("Chain shape (steps, walkers, parameters):", production_chain.shape)
print(f"Mean acceptance fraction: {emcee_sampler.acceptance_fraction.mean():.3f}")

### Recall the four-parameter posterior from Lecture 18

The pair plot shows the marginal distributions and correlations among the Ramsey parameters. Recreate this view before examining trace plots and formal diagnostics.


In [ ]:
# Flatten the production draws for the pair plot; retain the full chain for diagnostics.
flat_samples = emcee_sampler.get_chain(flat=True)
posterior_samples_df = pd.DataFrame(flat_samples, columns=parameter_names)

# Match the Lecture 18 corner plot: fine marginals, joint samples, and correlations.
four_parameter_correlation = posterior_samples_df[parameter_names].corr()
plot_samples_df = posterior_samples_df[parameter_names].iloc[::20]
parameter_labels = {
    'tau': r'$\tau$ ($\mu$s)',
    'contrast': r'$C$',
    'omega': r'$\omega$ (rad/$\mu$s)',
    'phase': r'$\phi$ (rad)',
}
fig, axes = plt.subplots(ndim, ndim, figsize=(8, 6), constrained_layout=True)
for row, row_name in enumerate(parameter_names):
    for column, column_name in enumerate(parameter_names):
        ax = axes[row, column]
        if row == column:
            ax.hist(posterior_samples_df[row_name], bins=45, density=True,
                    color='C0', alpha=0.75)
            ax.axvline(true_parameters[row_name], color='black', ls='--', lw=1)
        elif row > column:
            ax.scatter(plot_samples_df[column_name], plot_samples_df[row_name],
                       s=3, alpha=0.12, rasterized=True)
            ax.scatter(true_parameters[column_name], true_parameters[row_name],
                       marker='*', s=70, color='gold', edgecolor='black', zorder=3)
        else:
            corr = four_parameter_correlation.loc[row_name, column_name]
            ax.text(0.5, 0.5, f'r = {corr:.2f}', ha='center', va='center',
                    fontsize=13, transform=ax.transAxes)
            ax.set_axis_off()
        if row == ndim - 1:
            ax.set_xlabel(parameter_labels[column_name])
        else:
            ax.set_xticklabels([])
        if column == 0 and row != column:
            ax.set_ylabel(parameter_labels[row_name])
        elif column != 0:
            ax.set_yticklabels([])
fig.suptitle('Ramsey posterior: Lecture 18 reminder', y=1.02)
plt.show()

An `emcee` chain is stored with shape `(steps, walkers, parameters)`. Each walker moves through parameter space one proposal at a time, so neighboring saved positions are usually similar. The ensemble move also couples walkers. The acceptance fraction tells us how often proposals were accepted, but a high or low value alone does not tell us whether the posterior was explored well.


In [ ]:
# Plot a subset of walker paths so slow movement and sticking are visible.
fig, axes = plt.subplots(ndim, 1, figsize=(10, 8), sharex=True, constrained_layout=True)
for parameter_index, ax in enumerate(axes):
    for walker_index in range(0, nwalkers, 12):
        ax.plot(production_chain[:, walker_index, parameter_index],
                lw=0.55, alpha=0.8, label=f"walker {walker_index}")
    ax.axhline(true_parameters[parameter_names[parameter_index]],
               color="black", ls="--", lw=1, label="generating value")
    ax.set_ylabel(parameter_names[parameter_index])
    ax.legend(fontsize=7, ncol=3)
axes[-1].set_xlabel("Production step")
plt.show()

# Summarize acceptance by walker rather than using it as a convergence verdict.
acceptance_df = pd.DataFrame({
    "walker": np.arange(nwalkers),
    "acceptance_fraction": emcee_sampler.acceptance_fraction,
})
display(acceptance_df.describe().loc[["mean", "min", "max"]])

### Why do we need diagnostics?

A finite run can look plausible and still give misleading summaries. Walkers may spend a long time near their starting positions, move so slowly that thousands of saved points contain little new information, or fail to visit a distant mode. A trace plot can reveal sticking or drift, but a visually steady trace is not proof that all important regions were found. Acceptance fraction is similarly limited: it summarizes proposals, not coverage of the posterior.

We therefore ask two different questions: (1) within a run, how much independent information did the correlated samples provide? (2) do independently initialized runs reach compatible parts of the target? The next sections introduce measures for each question.

### Worked failure example: a sampler that barely moves

Now keep the **same data, posterior, and starting walkers**, but choose a Gaussian proposal whose step sizes are much larger than the useful posterior region. The entries passed to `GaussianMove` are proposal **variances**, so the standard deviations below are 2.0 μs in $\tau$, 0.6 in contrast, 1.2 rad/μs in $\omega$, and 3.0 rad in phase. 

The successful sampler run above uses the default "stretch move", whose proposal size depends on the current separation between walkers; it has no fixed per-parameter proposal standard deviation. The table below therefore compares the wide move's specified proposal standard deviations with the **empirical standard deviations of accepted parameter changes** in the successful run. These are different quantities, but the measured successful steps give a useful scale for seeing how extreme the wide proposal is. Many proposals will leave the prior bounds or land in a low-likelihood region. This run intentionally has no separate warmup; it is a diagnostic counterexample, not a posterior sample to report.

In [ ]:
# Deliberately use over-wide Gaussian proposals for the same Ramsey target.
proposal_sd = np.array([2.0, 0.6, 1.2, 3.0])

# The default stretch move has no fixed SD; measure accepted step sizes instead.
production_displacements = np.diff(production_chain, axis=0)
accepted_step_mask = np.any(production_displacements != 0, axis=2)
accepted_steps_df = pd.DataFrame(
    production_displacements[accepted_step_mask], columns=parameter_names
)
move_scale_comparison_df = pd.DataFrame({
    'parameter': parameter_names,
    'wide_Gaussian_proposal_SD': proposal_sd,
    'successful_accepted_step_SD': accepted_steps_df[parameter_names].std().to_numpy(),
})
display(move_scale_comparison_df)

In [ ]:
wide_move = emcee.moves.GaussianMove(proposal_sd**2)
wide_steps = 5000
np.random.seed(19193)
wide_sampler = emcee.EnsembleSampler(nwalkers, ndim, log_posterior, moves=wide_move)
wide_sampler.run_mcmc(initial_positions.copy(), wide_steps)
wide_chain = wide_sampler.get_chain()

# Compare accepted moves; a walker with zero accepts repeats its starting point.
comparison_df = pd.DataFrame({
    'sampler': ['over-wide Gaussian proposal', 'default emcee stretch move'],
    'steps_per_walker': [wide_steps, production_steps],
    'mean_acceptance_fraction': [wide_sampler.acceptance_fraction.mean(),
                                 emcee_sampler.acceptance_fraction.mean()],
    'walkers_with_no_accepted_move': [
        np.count_nonzero(wide_sampler.acceptance_fraction == 0),
        np.count_nonzero(emcee_sampler.acceptance_fraction == 0),
    ],
})
display(comparison_df)

# Compare the spread of saved values from the two runs, parameter by parameter.
# The failed-run SD is a diagnostic description, not a posterior uncertainty.
wide_samples_df = pd.DataFrame(wide_sampler.get_chain(flat=True),
                               columns=parameter_names)
successful_sd = posterior_samples_df[parameter_names].std()
failed_sd = wide_samples_df[parameter_names].std()
spread_comparison_df = pd.DataFrame({
    'parameter': parameter_names,
    'successful_sample_SD': successful_sd.to_numpy(),
    'failed_sample_SD': failed_sd.to_numpy(),
    'failed_to_successful_SD_ratio': (failed_sd / successful_sd).to_numpy(),
})
display(spread_comparison_df)

# Compare traces over the same number of saved steps for every parameter.
fig, axes = plt.subplots(ndim, 2, figsize=(11, 8), sharex=True,
                         constrained_layout=True)
for parameter_index, name in enumerate(parameter_names):
    for column, chain in enumerate((wide_chain, production_chain[:wide_steps])):
        ax = axes[parameter_index, column]
        for walker_index in range(0, nwalkers, 12):
            ax.plot(chain[:, walker_index, parameter_index], lw=0.65,
                    alpha=0.8)
        ax.axhline(true_parameters[name], color='black', ls='--', lw=1)
        ax.set_ylabel(name)
axes[0, 0].set_title('Over-wide proposal: unreliable run')
axes[0, 1].set_title('Default emcee: post-warmup production')
for ax in axes[-1]:
    ax.set_xlabel('Saved step')
plt.show()


**What failed?** In the left column, repeated flat segments show walkers remaining at the same position while proposals are rejected. The table counts walkers that never accepted a proposal and makes the very low acceptance fraction explicit. The right column shows the earlier production run for comparison. The dashed generating values are known only because these data are synthetic; in a real project, use the traces and between-run checks rather than agreement with an unknown truth.

The failed run's $5000\times48$ saved positions are **not** 240,000 independent measurements of the posterior. Repeating a starting value does not add information. The standard-deviation table compares the empirical spreads of saved values; the failed-run values are not trustworthy estimates of posterior uncertainty. Its histogram or median might even look plausible because the starts were chosen near the generating values, but that appearance does not validate uncertainty intervals. Fix the proposal scale or use a suitable move, allow warmup, run long enough, and reassess traces, autocorrelation, ESS, and independent ensembles before reporting results. There is no universal acceptance-fraction cutoff that proves success; the failure here is the combination of near-zero acceptance and frozen traces.

## Part 3: Autocorrelation, effective sample size, and Monte Carlo error

MCMC samples are correlated because each new position is proposed from the walker's current position. **Autocorrelation** measures how much a parameter value resembles values a certain number of steps earlier. Strong, slowly fading autocorrelation means nearby saved draws repeat similar information. For a parameter, the **integrated autocorrelation time** $\tau_{\mathrm{int}}$ summarizes this correlation across lags; roughly, one effectively independent draw is obtained per $\tau_{\mathrm{int}}$ steps of a walker.

The **effective sample size (ESS)** is the approximate number of independent draws that would give the same precision for estimating a posterior mean as the correlated chain. For an ensemble chain, a useful estimate is

$$N_{\mathrm{eff}} \approx \frac{N_{\mathrm{step}}N_{\mathrm{walker}}}{\tau_{\mathrm{int}}}.$$

Thus a chain can contain many saved states but have a much smaller ESS. ESS is parameter-specific because different parameters can mix at different rates. It measures sampling precision, not the width of the posterior: posterior standard deviation describes uncertainty in the parameter, while ESS tells how accurately the finite run estimates summaries of that posterior.

The **Monte Carlo standard error (MCSE)** estimates the run-to-run numerical noise in a posterior mean. If the posterior sample standard deviation is $s$, a rough estimate is $s/\sqrt{N_{\mathrm{eff}}}$. A large MCSE suggests running longer; a small MCSE means the estimated mean is numerically stable to the desired precision, but does not prove that the sampler found every mode. `emcee` estimates $\tau_{\mathrm{int}}$ from the chain. Short runs can produce unreliable estimates, so compare the run length with the estimated autocorrelation time and check whether estimates stabilize as more steps are included.

In [ ]:
# Use emcee's integrated-autocorrelation estimator on the full ensemble chain.
# tol=0 returns an estimate even for short teaching runs; interpret cautiously.
integrated_time = emcee_sampler.get_autocorr_time(tol=0)
flat_samples = emcee_sampler.get_chain(flat=True)
posterior_samples_df = pd.DataFrame(flat_samples, columns=parameter_names)

# Convert the estimated autocorrelation time into ESS and Monte Carlo error.
number_of_draws = production_steps * nwalkers
ess_values = number_of_draws / integrated_time
posterior_sd = posterior_samples_df[parameter_names].std().to_numpy()
mcse_values = posterior_sd / np.sqrt(ess_values)
autocorr_summary_df = pd.DataFrame({
    "parameter": parameter_names,
    "integrated_autocorrelation_time": integrated_time,
    "effective_sample_size": ess_values,
    "posterior_standard_deviation": posterior_sd,
    "mean_MCSE": mcse_values,
})
display(autocorr_summary_df)

### How precise are these posterior summaries?

The table's `mean_MCSE` is about **1–2% of the posterior standard deviation** for each parameter. That ratio describes the numerical error in our estimate of the *posterior mean*; it does not directly measure the error in the estimated *posterior standard deviation*. For roughly Gaussian draws, a rough fractional Monte Carlo error on a sample standard deviation is $1/\sqrt{2N_{\mathrm{eff}}}$. With $N_{\mathrm{eff}}$ around 5,000, this is about **1%** if the effective sample size for the spread is similar to the one estimated for the mean. Correlation in squared deviations can make the actual error different, so treat this as a scale estimate rather than a precise error bar.

Both numerical errors appear small compared with the posterior widths, so this run is probably sufficient for reporting these broad summaries **provided** the autocorrelation estimates stabilize and the independent-run checks agree. More steps can reduce Monte Carlo error; they do not shrink the posterior uncertainty set by the data and model. Activity 1 tests the first of those conditions.

### In-class coding activity 1 - Check whether ESS estimates stabilize

A **prefix** is the first $N$ consecutive production steps, keeping all walkers and parameters; for example, `production_chain[:1000]` is the 1,000-step prefix. A short prefix may give a noisy estimate of autocorrelation time. Use the first 1,000, first 2,500, and all production steps. For each prefix, ask `emcee` for $\tau_{\mathrm{int}}$ and calculate ESS for each parameter. Fill in the TODOs and compare the estimates. The prefixes overlap, so these are not independent estimates. Look for parameters whose estimated ESS changes substantially as the run grows; that instability is evidence the finite chain may not yet support a reliable precision estimate.

In [ ]:
# Compare autocorrelation and ESS estimates from cumulative chain prefixes.
prefix_lengths = [1000, 2500, production_steps]
prefix_diagnostic_rows = []
for prefix_length in prefix_lengths:
    prefix_chain = production_chain[:prefix_length]
    # TODO: Estimate integrated autocorrelation time for each parameter with emcee.
    prefix_tau_int = ...
    # TODO: Convert it to ESS using prefix_length and nwalkers.
    prefix_ess = ...
    for parameter_index, parameter_name in enumerate(parameter_names):
        prefix_diagnostic_rows.append({
            "production_steps": prefix_length,
            "parameter": parameter_name,
            "tau_int": prefix_tau_int[parameter_index],
            "ESS": prefix_ess[parameter_index],
        })
prefix_diagnostics_df = pd.DataFrame(prefix_diagnostic_rows)
prefix_diagnostics_df

## Part 4: Compare independent ensembles with split $\widehat R$

A single run may remain near its starting region or miss another posterior mode. To check between-run consistency, initialize several independent `emcee` ensembles at dispersed points and compare their sampled values. Walkers within one ensemble are coupled by the sampler's move, so they are not independent chains for this comparison. Here we use one designated walker from each of four separately initialized ensembles; each selected trajectory comes from a different run.

**Split $\widehat R$**, called the *split potential scale reduction factor* (or split R-hat), compares variation within trajectories with differences between trajectories. We calculate it **separately for each parameter**. Divide each of the $m$ independent trajectories into early and late halves, giving $M=2m$ half-chains of $n$ draws each. Splitting lets the diagnostic detect a change between the early and late parts of a run.

For one parameter, let $\bar\theta_j$ and $s_j^2$ be the sample mean and sample variance of half-chain $j$, and let $\bar\theta$ be the mean of all $M$ half-chain means. Define the **within-chain variance** $W$ and **between-chain variance** $B$ by

$$
W=\frac{1}{M}\sum_{j=1}^{M}s_j^2,
\qquad
B=\frac{n}{M-1}\sum_{j=1}^{M}(\bar\theta_j-\bar\theta)^2.
$$

The **estimated marginal variance** $\widehat V$ combines these two estimates. The split potential scale reduction factor $\widehat R$ is the ratio of the estimated marginal standard deviation to the within-chain standard deviation:

$$
\widehat V=\frac{n-1}{n}W+\frac{B}{n},
\qquad
\widehat R=\sqrt{\frac{\widehat V}{W}}.
$$

In the code, `within`, `between`, and `variance_estimate` are $W$, $B$, and $\widehat V$. If independent runs and their early and late halves explore the same posterior, $\widehat R$ should be close to 1. A value appreciably above 1 suggests incomplete mixing or a remaining dependence on starting position. A value near 1 is reassuring, but it cannot prove that every posterior mode was found: all runs could miss the same region. This is the basic variance-based calculation; research analyses often use a rank-normalized version and examine traces alongside it.

In [ ]:
# Compute basic split-R-hat for chains shaped (chains, draws, parameters).
def split_rhat(chains):
    chains = np.asarray(chains)
    n_chains, n_draws, n_parameters = chains.shape
    half = n_draws // 2
    split = np.concatenate([chains[:, :half], chains[:, -half:]], axis=0)
    n = split.shape[1]
    within = np.mean(np.var(split, axis=1, ddof=1), axis=0)
    between = n * np.var(np.mean(split, axis=1), axis=0, ddof=1)
    variance_estimate = ((n - 1) / n) * within + between / n
    return np.sqrt(variance_estimate / within)


# Start four independent ensembles in dispersed, valid regions of the prior.
independent_centers = np.array([
    [1.0, 0.30, true_parameters["omega"] - 0.15, -0.20],
    [1.8, 0.55, true_parameters["omega"] + 0.10, 0.10],
    [4.2, 0.65, true_parameters["omega"] - 0.05, 0.45],
    [5.0, 0.90, true_parameters["omega"] + 0.12, -0.50],
])
independent_ensembles = []
independent_full_chains = []
ensemble_acceptance = []
independent_walkers = 24
independent_steps = 3000
independent_warmup = 500
start_scales = np.array([0.12, 0.035, 0.06, 0.12])

for run_id, center in enumerate(independent_centers):
    # Seed each ensemble independently and generate valid starting walkers.
    np.random.seed(1930 + run_id)
    run_rng = np.random.default_rng(2930 + run_id)
    run_positions = center + run_rng.normal(
        size=(independent_walkers, ndim)
    ) * start_scales
    run_sampler = emcee.EnsembleSampler(
        independent_walkers, ndim, log_posterior
    )
    run_state = run_sampler.run_mcmc(run_positions, independent_warmup)
    run_sampler.reset()
    run_sampler.run_mcmc(run_state, independent_steps)

    # Retain the full production ensemble for the pair plot and one walker for R-hat.
    run_chain = run_sampler.get_chain()
    independent_full_chains.append(run_chain)
    independent_ensembles.append(run_chain[:, 0, :])
    ensemble_acceptance.append(run_sampler.acceptance_fraction.mean())

independent_ensemble_chains = np.stack(independent_ensembles)
print("Independent selected-walker shape (chains, steps, parameters):",
      independent_ensemble_chains.shape)
print("Mean acceptance by ensemble:", np.round(ensemble_acceptance, 3))

In [ ]:
# Plot the selected walker from each independent ensemble for a visual check.
fig, axes = plt.subplots(ndim, 1, figsize=(10, 8), sharex=True, constrained_layout=True)
for parameter_index, ax in enumerate(axes):
    for chain_index in range(independent_ensemble_chains.shape[0]):
        ax.plot(independent_ensemble_chains[chain_index, :, parameter_index],
                lw=0.6, label=f"independent ensemble {chain_index + 1}")
    ax.set_ylabel(parameter_names[parameter_index])
    ax.legend(fontsize=8, ncol=2)
axes[-1].set_xlabel("Production step")
plt.show()

### Where did the four ensembles start, and where did they finish?

The **lower triangle** of the pair plot places a star at each run's starting center and overlays its retained production draws in the same color. These broad axes show how far apart the initial centers were. The **upper triangle** repeats each parameter pair using only production draws, so the axes zoom in on the posterior: overlapping colored clouds indicate that the separate runs reached similar joint distributions. The diagonal panels compare their one-parameter distributions. Each cloud includes draws from all walkers in that ensemble, while split $\widehat R$ below still uses one designated walker from each independent run. Visual overlap is useful evidence, but the trace and $\widehat R$ checks remain necessary.


In [ ]:
# Convert a thin subset of each full production ensemble to a DataFrame for plotting.
centers_df = pd.DataFrame(independent_centers, columns=parameter_names)
run_samples = [
    pd.DataFrame(chain[::50].reshape(-1, ndim), columns=parameter_names)
    for chain in independent_full_chains
]
run_colors = plt.get_cmap('tab10').colors[:len(run_samples)]

# Lower panels show starts plus retained draws; upper panels zoom in on draws.
fig, axes = plt.subplots(ndim, ndim, figsize=(10, 9), constrained_layout=True)
for row, row_name in enumerate(parameter_names):
    for column, column_name in enumerate(parameter_names):
        ax = axes[row, column]
        for run_id, samples_df in enumerate(run_samples):
            color = run_colors[run_id]
            if row == column:
                ax.hist(samples_df[row_name], bins=35, density=True,
                        histtype='step', lw=1.4, color=color)
            else:
                ax.scatter(samples_df[column_name], samples_df[row_name],
                           s=3, alpha=0.07, color=color, rasterized=True)
                if row > column:
                    ax.scatter(centers_df.loc[run_id, column_name],
                               centers_df.loc[run_id, row_name], marker='*',
                               s=100, color=color, edgecolor='black', zorder=5)
        if row == ndim - 1:
            ax.set_xlabel(column_name)
        else:
            ax.set_xticklabels([])
        if column == 0 and row != column:
            ax.set_ylabel(row_name)
        else:
            ax.set_yticklabels([])

# Colors identify independent runs; stars identify their starting centers.
from matplotlib.lines import Line2D
legend_handles = [
    Line2D([0], [0], marker='o', linestyle='none', color=run_colors[i],
           label=f'ensemble {i + 1}')
    for i in range(len(run_samples))
]
fig.legend(handles=legend_handles, loc='upper center', ncol=4,
           bbox_to_anchor=(0.5, 1.04))
fig.suptitle('Independent starts and overlapping Ramsey posteriors', y=1.08)
plt.show()


### In-class coding activity 2 - Diagnose independent runs

Use the objects built in Part 4; you do not need to rerun the sampler or rewrite the $\widehat R$ formula.

1. Confirm the shape of `independent_ensemble_chains`: the axes are **independent runs, production steps, parameters**. Each run contributes one designated walker trajectory. Warmup was already discarded by `run_sampler.reset()`.
2. Pass that three-dimensional array to the completed `split_rhat(chains)` function above. It returns one value for each column in `parameter_names`.
3. Pair those values with the parameter names in the provided table. Identify the parameter with the largest split $\widehat R$.
4. Compare the $\widehat R$ table with the separate mean-acceptance table. Do the four runs appear consistent? What could a value near one still fail to detect?


In [ ]:
# Step 1: Check that the axes match the input expected by split_rhat.
print('Shape (independent runs, production steps, parameters):',
      independent_ensemble_chains.shape)
print('Parameter order:', parameter_names)

# Step 2: Reuse the completed Part 4 function on the four selected trajectories.
# TODO: Pass independent_ensemble_chains to split_rhat; keep one result per parameter.
rhat_values = ...

# Step 3: Associate each result with its parameter name.
rhat_summary_df = pd.DataFrame({
    'parameter': parameter_names,
    'split_Rhat': rhat_values,
})
display(rhat_summary_df)

# TODO: Find the row index of the largest split R-hat (hint: .idxmax()).
largest_rhat_index = ...
display(rhat_summary_df.loc[[largest_rhat_index]])

# Step 4: Compare with acceptance, which does not by itself establish convergence.
ensemble_summary_df = pd.DataFrame({
    'ensemble': np.arange(1, len(ensemble_acceptance) + 1),
    'mean_acceptance_fraction': ensemble_acceptance,
})
display(ensemble_summary_df)


## Part 5: Apply the workflow to Project 2 and HPC runs

Project 2 uses `emcee` for a correlated, multi-parameter Pantheon+ posterior. The same failure modes matter there, but a longer runtime does not by itself guarantee a reliable result. Use the diagnostics to justify that the chains explored the target and that numerical sampling error is small compared with the posterior uncertainty. The Ramsey example is fast enough to practice the workflow:

- inspect traces and acceptance behavior for every parameter;
- estimate autocorrelation time and ESS, and check whether the production run is long relative to those timescales;
- compare independently initialized runs and report split $\widehat R$;
- extend or retune runs when the diagnostics show inadequate exploration; and
- report numerical sampling precision separately from the posterior's physical/statistical width.



For the Lecture 18 Ramsey benchmark, first create a four-parameter saved chain, then diagnose it from the repository root:

```bash
python scripts/lecture18/run_mcmc.py --dimensions 4 --steps 5000 --warmup 1000 --chain-id 0
python scripts/lecture19/diagnose_emcee.py --input outputs/ramsey_emcee_4d_chain0.npz --output outputs/lecture19_diagnostics.csv
```

To estimate split $\widehat R$, create at least one more independently seeded ensemble with the **same data seed**, then provide both files:

```bash
python scripts/lecture18/run_mcmc.py --dimensions 4 --steps 5000 --warmup 1000 --chain-id 1
python scripts/lecture19/diagnose_emcee.py --input outputs/ramsey_emcee_4d_chain0.npz --input outputs/ramsey_emcee_4d_chain1.npz --output outputs/lecture19_diagnostics.csv
```

The first command produces the file consumed by the second. Repeated `--input` options must refer to independently generated ensemble files, not repeated copies of one file. The Lecture 18 runner changes the random walker positions and sampler seed for each `--chain-id`, but keeps the same nominal starting center. Part 4 used deliberately dispersed centers; that stronger check can reveal dependence on initialization that the default script runs might miss. The script uses one walker from each independent ensemble for this between-run comparison; it never treats walkers within an ensemble as independent chains. With more than one input, the helper also saves split $\widehat R$ to `outputs/lecture19_diagnostics_rhat.csv`. The command-line helper expects the Lecture 18 NPZ layout: `chain` shaped `(steps, walkers, parameters)`, plus `parameter_names` and `acceptance_fraction`. If the Project 2 script writes a different layout, adapt its loading step; the same `emcee` diagnostics can be called directly on its sampler chain.

The Project 2 model and data are more expensive than this Ramsey example. Use the project's own command-line script, benchmark a short run, then submit longer jobs using the SciClone batch workflow from Lectures 9 and 18. After each run, use the diagnostics to decide whether to extend sampling, change initialization, or investigate a possible missed region; do not report a posterior interval based only on a successful script exit. Request resources that match the number of worker processes; do not run long jobs on a login node. The Lecture 18 `run_emcee.slurm` file is a Ramsey example to adapt, not a substitute for Project 2's inference script. Keep the data seed fixed across sampler comparisons and vary sampler seeds across independent runs.

## Conclusion and Project 2 checklist

- A sampler can return many saved positions while almost never moving; the over-wide proposal example shows why acceptance and traces matter.
- `emcee` stores steps, walkers, and parameters in separate axes; preserve that structure until the diagnostic needs a different view.
- Integrated autocorrelation time, ESS, and MCSE describe finite-run precision, while posterior width describes parameter uncertainty.
- Independent ensembles are needed for between-run comparisons; walkers within one ensemble are coupled.
- No single diagnostic proves convergence. Combine traces, autocorrelation, ESS, acceptance behavior, and split $\widehat R$ before interpreting intervals.
- Project 2 asks you to use these checks on a higher-dimensional physics inference problem and document your decisions.

## References and further reading

- [Unit 4 course resources](../resources/unit-4-monte-carlo-inference.md)
- [`emcee` autocorrelation analysis](https://emcee.readthedocs.io/en/stable/tutorials/autocorr/)
- [`emcee` monitoring and diagnostics](https://emcee.readthedocs.io/en/stable/user/monitor/)
- [Stan Reference Manual: MCMC diagnostics](https://mc-stan.org/docs/reference-manual/analysis.html)
- [Project 2 template](https://github.com/WM-PHYS690-Fall2026/project2-template)